# 02 — Linear Attention

Linear attention uses a kernel trick to reduce the O(n²) attention matrix to O(n) operations.

## Kernel Trick for Attention and FAVOR+

Standard attention: *softmax(QK^T/√d)V*

The key observation (Katharopoulos et al., 2020): if we replace softmax with a feature map *φ* such that *softmax(q,k) ≈ φ(q)^T φ(k)*, we can rewrite:

$$\text{Attn}(q_i) = \frac{\sum_j φ(q_i)^T φ(k_j) v_j}{\sum_j φ(q_i)^T φ(k_j)} = \frac{φ(q_i)^T \sum_j φ(k_j) v_j^T}{φ(q_i)^T \sum_j φ(k_j)}$$

The key: compute the matrix products in a different order. $(\sum_j φ(k_j) v_j^T)$ is *d×d* and can be computed once for all query positions — O(nd²) total instead of O(n²d).

**FAVOR+** (Performers, Choromanski et al., 2021) uses random Fourier features as the kernel approximation:
$$φ(x) = \frac{1}{\sqrt{m}}[e^{ω_1^T x - ||x||^2/2}, ..., e^{ω_m^T x - ||x||^2/2}]$$
where *ω_i ~ N(0,I)*. This approximates the softmax kernel while remaining unbiased and maintaining O(n) memory.

**Quality tradeoff**: linear attention is not identical to softmax attention. The approximation error grows when the attention matrix is highly peaked (sparse attention patterns are harder to approximate). For most NLP tasks, the degradation is small; for tasks requiring sharp, localised attention (e.g., copying exact tokens), quality drops more.

In [ ]:
# Linear attention implementation + quality tradeoff
import numpy as np
import torch
import torch.nn as nn

class StandardAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2,-1)) / self.d_head**0.5
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1,2).reshape(B, N, D)
        return self.out(out)


class LinearAttention(nn.Module):
    """Linear attention using ELU feature map."""
    def __init__(self, d_model, n_heads, eps=1e-6):
        super().__init__()
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.eps = eps

    def feature_map(self, x):
        return torch.nn.functional.elu(x) + 1  # positive, simple kernel

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head).permute(2,0,3,1,4)
        q, k, v = self.feature_map(qkv[0]), self.feature_map(qkv[1]), qkv[2]

        # O(n) linear attention: compute KV first
        # k: (B, H, N, d); v: (B, H, N, d)
        kv = torch.einsum('bhnk,bhnv->bhkv', k, v)  # (B, H, d, d)
        k_sum = k.sum(dim=2)  # (B, H, d)

        # q: (B, H, N, d)
        qkv_out = torch.einsum('bhnk,bhkv->bhnv', q, kv)  # (B, H, N, d)
        q_k_sum = torch.einsum('bhnk,bhk->bhn', q, k_sum).unsqueeze(-1)  # (B, H, N, 1)

        out = qkv_out / (q_k_sum + self.eps)
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.out(out)

# Quality comparison
torch.manual_seed(42)
d_model, n_heads = 64, 4
std_attn = StandardAttention(d_model, n_heads)
lin_attn = LinearAttention(d_model, n_heads)

# Copy weights for fair comparison
lin_attn.qkv.weight.data = std_attn.qkv.weight.data.clone()
lin_attn.out.weight.data = std_attn.out.weight.data.clone()

x = torch.randn(2, 128, d_model)
out_std = std_attn(x)
out_lin = lin_attn(x)

diff = (out_std - out_lin).abs().mean().item()
print(f'Mean absolute difference (std vs linear attention): {diff:.4f}')
print(f'Relative error: {diff/out_std.abs().mean().item():.3f}')

In [ ]:
# Speed comparison
import time
import matplotlib.pyplot as plt

seq_lens = [64, 128, 256, 512, 1024, 2048]
std_times, lin_times = [], []

for n in seq_lens:
    x_test = torch.randn(1, n, 64)

    # Standard
    t0 = time.perf_counter()
    for _ in range(20):
        with torch.no_grad():
            _ = std_attn(x_test)
    std_times.append((time.perf_counter() - t0) / 20 * 1000)

    # Linear
    t0 = time.perf_counter()
    for _ in range(20):
        with torch.no_grad():
            _ = lin_attn(x_test)
    lin_times.append((time.perf_counter() - t0) / 20 * 1000)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(seq_lens, std_times, 'o-', label='Standard O(n²)', color='tomato')
ax.plot(seq_lens, lin_times, 's-', label='Linear O(n)', color='steelblue')
ax.set_xlabel('Sequence length')
ax.set_ylabel('Time per call (ms)')
ax.set_title('Standard vs Linear Attention Speed')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/linear_attn_speed.png', dpi=100, bbox_inches='tight')
plt.show()
print('Speed comparison saved')